# Course Generation Algorithm — Hackable Notebook

Reimplements Lectura's lecture-generation pipeline (**handbook course → lecture plan → per-lecture outline/slides/Bloom**)
as a plain, editable script for prompt-engineering experiments.

- **Step 0** loads the MSc handbook PDF and lets you pick a course section (default: **4.1 Software Engineering Modules**, from page 23).
- **Step 0.5** extracts or generates **10–12 lectures**, saves JSON + Markdown.
- **Steps 1–3** define outline / slides / Bloom prompts and generators.
- **Step 4** loops over every lecture in `LECTURE_PLAN`, runs outline → slides → Bloom, saves per-lecture JSON and a combined PDF report.
- One `call_llm()` function talks to **OpenRouter** (`OPENROUTER_API_KEY`).

Run cells top to bottom.


In [1]:
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import requests
from IPython.display import HTML, display

NOTEBOOK_DIR = Path.cwd()


def _load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


_load_dotenv(NOTEBOOK_DIR / ".env")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
LLM_PROVIDER = "openrouter" if OPENROUTER_API_KEY else "anthropic" if ANTHROPIC_API_KEY else ""
if not LLM_PROVIDER:
    raise EnvironmentError(
        "Set OPENROUTER_API_KEY or ANTHROPIC_API_KEY in the environment or .env file."
    )

# OpenRouter slug; Anthropic fallback maps this to ANTHROPIC_TEXT_MODEL.
MODEL = "anthropic/claude-sonnet-4.5"
print(f"LLM provider: {LLM_PROVIDER} | model: {MODEL}")

## Step 0 — Course selection from CSSE Handbook

Reads `CSSE_MSc_Handbook_ Fall 2025_V1_3.pdf` from the project root and extracts the selected course section.

**Change your course:** edit `SELECTED_COURSE_ID` (and optionally `SELECTED_MODULE_CODE`) in the next cell, then re-run it.

| ID | Section | Pages |
|----|---------|-------|
| `4.1` | Software Engineering Modules | 23–35 |
| `4.2` | Cybersecurity Modules | 36–49 |
| `4.3` | Artificial Intelligence Modules | 50–59 |

More sections can be added to `HANDBOOK_SECTIONS` as you expand experiments.

In [ ]:
# --- Handbook & course selection (edit these) ---
HANDBOOK_FILENAME = "CSSE_MSc_Handbook_ Fall 2025_V1_3.pdf"
SELECTED_COURSE_ID = "4.1"  # e.g. "4.2" Cybersecurity, "4.3" AI
SELECTED_MODULE_CODE = None  # e.g. "2025-MCSSE-SE-01"; None = first module in section

HANDBOOK_SECTIONS = {
    "4.1": {
        "title": "Software Engineering Modules",
        "start_page": 23,
        "end_page": 35,
    },
    "4.2": {
        "title": "Cybersecurity Modules",
        "start_page": 36,
        "end_page": 49,
    },
    "4.3": {
        "title": "Artificial Intelligence Modules",
        "start_page": 50,
        "end_page": 59,
    },
}


def _ensure_pypdf():
    try:
        from pypdf import PdfReader  # type: ignore
        return PdfReader
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdf", "-q"])
        from pypdf import PdfReader  # type: ignore
        return PdfReader


def extract_handbook_pages(pdf_path: Path, start_page: int, end_page: int) -> str:
    """Extract text from inclusive 1-based PDF page range."""
    PdfReader = _ensure_pypdf()
    reader = PdfReader(str(pdf_path))
    chunks: list[str] = []
    for page_num in range(start_page, min(end_page, len(reader.pages)) + 1):
        text = reader.pages[page_num - 1].extract_text() or ""
        chunks.append(f"--- Page {page_num} ---\n{text.strip()}")
    return "\n\n".join(chunks)


def parse_handbook_modules(section_text: str) -> list[dict]:
    """Parse module blocks from a handbook section (Module Name / Module Code / ECTS)."""
    modules: list[dict] = []
    for match in re.finditer(
        r"Module Name\s+(.+?)\s+Module Code\s+(\S+)\s+Module ECTS\s+(\d+)",
        section_text,
        flags=re.DOTALL,
    ):
        name = re.sub(r"\s+", " ", match.group(1)).strip()
        modules.append(
            {
                "name": name,
                "code": match.group(2),
                "ects": int(match.group(3)),
            }
        )
    return modules


handbook_path = NOTEBOOK_DIR / HANDBOOK_FILENAME
if not handbook_path.exists():
    raise FileNotFoundError(
        f"Handbook not found: {handbook_path}\n"
        "Place CSSE_MSc_Handbook_ Fall 2025_V1_3.pdf in the project root."
    )

if SELECTED_COURSE_ID not in HANDBOOK_SECTIONS:
    raise ValueError(
        f"Unknown SELECTED_COURSE_ID={SELECTED_COURSE_ID!r}. "
        f"Choose one of: {', '.join(HANDBOOK_SECTIONS)}"
    )

selected_section = HANDBOOK_SECTIONS[SELECTED_COURSE_ID]
handbook_section_text = extract_handbook_pages(
    handbook_path,
    selected_section["start_page"],
    selected_section["end_page"],
)
handbook_modules = parse_handbook_modules(handbook_section_text)

if not handbook_modules:
    raise RuntimeError(
        f"No modules parsed in section {SELECTED_COURSE_ID}. "
        "Check page range or PDF text extraction."
    )

if SELECTED_MODULE_CODE:
    selected_module = next(
        (m for m in handbook_modules if m["code"] == SELECTED_MODULE_CODE),
        None,
    )
    if not selected_module:
        raise ValueError(
            f"Module {SELECTED_MODULE_CODE!r} not found. "
            f"Available: {[m['code'] for m in handbook_modules]}"
        )
else:
    selected_module = handbook_modules[0]

COURSE_TITLE = f"{SELECTED_COURSE_ID} {selected_section['title']}"
COURSE_CONTEXT = (
    f"MSc CSSE handbook section: {COURSE_TITLE}\n"
    f"Selected module: {selected_module['name']} ({selected_module['code']}, "
    f"{selected_module['ects']} ECTS)\n\n"
    f"{handbook_section_text[:12000]}"
)

display(HTML(f"<h3>Handbook: {HANDBOOK_FILENAME}</h3>"))
display(
    HTML(
        "<dl>"
        f"<dt><b>Course section</b></dt><dd>{COURSE_TITLE}</dd>"
        f"<dt><b>Pages</b></dt><dd>{selected_section['start_page']}–{selected_section['end_page']}</dd>"
        f"<dt><b>Selected module</b></dt><dd>{selected_module['name']} ({selected_module['code']})</dd>"
        "</dl>"
    )
)
rows = "".join(
    f"<tr><td>{m['code']}</td><td>{m['name']}</td><td>{m['ects']}</td></tr>"
    for m in handbook_modules
)
display(
    HTML(
        "<table border=1 cellpadding=6 style='border-collapse:collapse'>"
        "<tr><th>Module code</th><th>Module name</th><th>ECTS</th></tr>"
        f"{rows}</table>"
    )
)
print(f"Using module: {selected_module['code']} — {selected_module['name']}")

## The one LLM call

Every pipeline step funnels through `call_llm()`. Change the provider/model here and it propagates everywhere.

In [2]:
JSON_ONLY_SUFFIX = (
    "\n\nYou MUST respond with valid JSON only — no markdown, "
    "no code fences, no commentary before or after."
)


def _anthropic_model_name(model: str) -> str:
    if os.environ.get("ANTHROPIC_TEXT_MODEL"):
        return os.environ["ANTHROPIC_TEXT_MODEL"]
    slug = model.split("/")[-1] if "/" in model else model
    if "claude-sonnet-4" in slug:
        return "claude-sonnet-4-5-20250929"
    return slug


def call_llm(
    system_prompt: str,
    user_prompt: str,
    *,
    temperature: float = 0.5,
    max_tokens: int = 8192,
    model: str = MODEL,
) -> str:
    """Single entry point for every LLM call in this notebook."""
    print(f"    -> calling {model} via {LLM_PROVIDER} (temperature={temperature}, max_tokens={max_tokens})...")
    start = time.time()
    system = system_prompt + JSON_ONLY_SUFFIX

    if LLM_PROVIDER == "openrouter":
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": model,
                "temperature": temperature,
                "max_tokens": max_tokens,
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user", "content": user_prompt},
                ],
            },
            timeout=180,
        )
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"]
    else:
        anthropic_model = _anthropic_model_name(model)
        response = requests.post(
            "https://api.anthropic.com/v1/messages",
            headers={
                "x-api-key": ANTHROPIC_API_KEY,
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": anthropic_model,
                "max_tokens": max_tokens,
                "temperature": temperature,
                "system": system,
                "messages": [{"role": "user", "content": user_prompt}],
            },
            timeout=180,
        )
        response.raise_for_status()
        payload = response.json()
        content = "".join(
            block.get("text", "")
            for block in payload.get("content", [])
            if block.get("type") == "text"
        )

    elapsed = time.time() - start
    print(f"    <- received response in {elapsed:.1f}s ({len(content)} chars)")
    return _strip_json_fences(content)


def _strip_json_fences(text: str) -> str:
    text = text.strip()
    match = re.match(r"^```(?:json)?\s*([\s\S]*?)```$", text)
    if match:
        return match.group(1).strip()
    match = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if match:
        return match.group(1).strip()
    return text


def parse_with_retry(generate_fn, max_retries: int = 3, label: str = "") -> dict:
    """Calls generate_fn() (returns a JSON string), retrying by re-calling the LLM on parse failure."""
    prefix = f"[{label}] " if label else ""
    last_error = None
    raw = None
    for attempt in range(max_retries):
        print(f"{prefix}attempt {attempt + 1}/{max_retries}")
        raw = generate_fn()
        try:
            result = json.loads(raw)
            print(f"{prefix}parsed valid JSON on attempt {attempt + 1}")
            return result
        except json.JSONDecodeError as e:
            last_error = e
            print(f"{prefix}attempt {attempt + 1} failed to parse JSON: {e}")
    raise RuntimeError(
        f"Failed to parse JSON after {max_retries} attempts: {last_error}\nLast raw output:\n{raw}"
    )


## Minimal HTML visualizations

No charting libraries — arrays render as `<ul>` lists, tabular data as plain `<table>`.

In [3]:
def show_title(text: str, level: int = 4) -> None:
    display(HTML(f"<h{level}>{text}</h{level}>"))


def show_list(items: list, title: str | None = None) -> None:
    html = f"<b>{title}</b>" if title else ""
    html += "<ul>" + "".join(f"<li>{item}</li>" for item in items) + "</ul>"
    display(HTML(html))


def show_table(rows: list[list], headers: list[str]) -> None:
    html = "<table border=1 cellpadding=6 style=\'border-collapse:collapse\'><tr>"
    html += "".join(f"<th>{h}</th>" for h in headers) + "</tr>"
    for row in rows:
        html += "<tr>" + "".join(f"<td>{c}</td>" for c in row) + "</tr>"
    html += "</table>"
    display(HTML(html))


def show_fields(d: dict, keys: list[str] | None = None) -> None:
    keys = keys or list(d.keys())
    html = "<dl>"
    for k in keys:
        v = d.get(k)
        if isinstance(v, list):
            v = ", ".join(map(str, v)) if v else "\u2014"
        html += f"<dt><b>{k}</b></dt><dd>{v}</dd>"
    html += "</dl>"
    display(HTML(html))


## Step 0.5 — Lecture plan (extract or generate)

Builds a **course-level lecture list** (target **10–12 lectures**) for the selected handbook module.

1. Try to **extract** lectures from handbook text (`Lecture 1`, `Week 2`, numbered sessions, …).
2. If fewer than `MIN_LECTURES_FROM_HANDBOOK` are found, **generate** a lecture plan with AI from `COURSE_CONTEXT`.
3. Save artifacts under `output/course-generation/`:
   - `{module-code}-lecture-plan.json`
   - `{module-code}-lecture-plan.md`
   - `{module-code}-generation-report.pdf` — running PDF log of algorithm output (lecture plan section for now)

Edit `TARGET_LECTURE_COUNT` (10–12) in the next cell if needed.

In [ ]:
from datetime import datetime, timezone

# --- Lecture plan settings (edit these) ---
TARGET_LECTURE_COUNT = 10  # use 10–12
MIN_LECTURES_FROM_HANDBOOK = 8  # if extracted count is below this, generate instead
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "course-generation"

LECTURE_PLAN_SYSTEM_PROMPT = """You are an expert CS curriculum designer.
Design a university module delivery plan as a list of lectures.
Output ONLY valid JSON."""

LECTURE_PLAN_USER_PROMPT = """Design a lecture plan for this handbook module.

Module title: {module_title}
Module code: {module_code}
Target number of lectures: {target_count} (each about 90 minutes)

Handbook context:
{handbook_context}

Return JSON:
{{
  "source": "generated",
  "targetLectureCount": {target_count},
  "lectures": [
    {{
      "number": number,
      "title": string,
      "durationMinutes": 90,
      "summary": string,
      "topics": string[]
    }}
  ]
}}

Rules:
- Produce exactly {target_count} lectures in logical teaching order
- Base topics on module description and intended learning outcomes from the handbook
- Titles should be lecture-ready (clear, specific, MSc Software Engineering level)
- No duplicate or overlapping lecture titles"""


def _ensure_fpdf():
    try:
        from fpdf import FPDF  # type: ignore
        return FPDF
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "fpdf2", "-q"])
        from fpdf import FPDF  # type: ignore
        return FPDF


def pdf_safe(text: str) -> str:
    return text.encode("latin-1", errors="replace").decode("latin-1")


def module_slug(module_code: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "-", module_code).strip("-").lower()


def extract_module_text(section_text: str, module_code: str, module_name: str) -> str:
    """Narrow handbook text to the selected module block when possible."""
    pattern = rf"Module Name\s+{re.escape(module_name)}[\s\S]*?(?=Module Name\s+|\Z)"
    match = re.search(pattern, section_text, flags=re.IGNORECASE)
    if match:
        return match.group(0)
    if module_code in section_text:
        start = section_text.find(module_code)
        return section_text[start : start + 8000]
    return section_text


def extract_lectures_from_text(text: str) -> list[dict]:
    """Heuristic extraction of lecture/session lists from handbook-like text."""
    lectures: list[dict] = []
    seen_numbers: set[int] = set()

    labeled = re.finditer(
        r"(?:^|\n)\s*(?:Lecture|Session|Week|Unit)\s+(\d{1,2})\s*[\.:\-–—]\s*(.+?)(?=(?:\n\s*(?:Lecture|Session|Week|Unit)\s+\d{1,2}\s*[\.:\-–—])|\Z)",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    for match in labeled:
        number = int(match.group(1))
        if number in seen_numbers:
            continue
        title = re.sub(r"\s+", " ", match.group(2)).strip(" .")
        title = re.split(r"\n\s*(?:Duration|Target|Primary goal|Module|Assessment)\b", title, maxsplit=1)[0].strip()
        if len(title) < 8:
            continue
        seen_numbers.add(number)
        lectures.append(
            {
                "number": number,
                "title": title,
                "durationMinutes": 90,
                "summary": "",
                "topics": [],
            }
        )

    if len(lectures) < MIN_LECTURES_FROM_HANDBOOK:
        numbered = re.finditer(r"(?:^|\n)\s*(\d{1,2})\.\s+([A-Z][^\n]{12,140})", text)
        for match in numbered:
            number = int(match.group(1))
            if number in seen_numbers or number > 20:
                continue
            title = re.sub(r"\s+", " ", match.group(2)).strip()
            if any(skip in title.lower() for skip in ("module", "assessment", "ects", "cp")):
                continue
            seen_numbers.add(number)
            lectures.append(
                {
                    "number": number,
                    "title": title,
                    "durationMinutes": 90,
                    "summary": "",
                    "topics": [],
                }
            )

    lectures.sort(key=lambda item: item["number"])
    return lectures


def generate_lecture_plan(module: dict, handbook_context: str, target_count: int) -> dict:
    print(f"=== Step 0.5: generating {target_count} lectures for '{module['name']}' ===")

    def _call():
        user = LECTURE_PLAN_USER_PROMPT.format(
            module_title=module["name"],
            module_code=module["code"],
            target_count=target_count,
            handbook_context=handbook_context[:12000],
        )
        return call_llm(LECTURE_PLAN_SYSTEM_PROMPT, user, temperature=0.4)

    plan = parse_with_retry(_call, label="lecture-plan")
    plan["source"] = "generated"
    plan["targetLectureCount"] = target_count
    return plan


def build_lecture_plan(selected_module: dict, course_context: str, handbook_section_text: str, target_count: int) -> dict:
    module_text = extract_module_text(
        handbook_section_text,
        selected_module["code"],
        selected_module["name"],
    )
    extracted = extract_lectures_from_text(module_text)
    if len(extracted) >= MIN_LECTURES_FROM_HANDBOOK:
        print(f"=== Step 0.5: extracted {len(extracted)} lectures from handbook text ===")
        return {
            "source": "handbook",
            "targetLectureCount": len(extracted),
            "moduleCode": selected_module["code"],
            "moduleName": selected_module["name"],
            "lectures": extracted,
        }

    generated = generate_lecture_plan(selected_module, course_context, target_count)
    generated["moduleCode"] = selected_module["code"]
    generated["moduleName"] = selected_module["name"]
    return generated


def save_lecture_plan_json(plan: dict, path: Path) -> None:
    path.write_text(json.dumps(plan, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


def save_lecture_plan_markdown(plan: dict, path: Path) -> None:
    lines = [
        f"# Lecture plan — {plan.get('moduleName', 'Module')}",
        "",
        f"- **Module code:** {plan.get('moduleCode', '—')}",
        f"- **Source:** {plan.get('source', '—')}",
        f"- **Lecture count:** {len(plan.get('lectures', []))}",
        f"- **Generated at:** {datetime.now(timezone.utc).isoformat()}",
        "",
        "## Lectures",
        "",
    ]
    for lecture in plan.get("lectures", []):
        lines.append(f"### Lecture {lecture.get('number')}. {lecture.get('title')}")
        if lecture.get("durationMinutes"):
            lines.append(f"- **Duration:** {lecture['durationMinutes']} minutes")
        if lecture.get("summary"):
            lines.append(f"- **Summary:** {lecture['summary']}")
        topics = lecture.get("topics") or []
        if topics:
            lines.append("- **Topics:**")
            lines.extend(f"  - {topic}" for topic in topics)
        lines.append("")
    path.write_text("\n".join(lines), encoding="utf-8")


def save_generation_report_pdf(plan: dict, course_title: str, path: Path) -> None:
    FPDF = _ensure_fpdf()
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Helvetica", style="B", size=16)
    pdf.cell(0, 10, pdf_safe("Course Generation Algorithm — Report"), ln=True)
    pdf.set_font("Helvetica", size=11)
    pdf.multi_cell(0, 6, pdf_safe(f"Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}"))
    pdf.ln(2)
    pdf.set_font("Helvetica", style="B", size=13)
    pdf.cell(0, 8, pdf_safe("Step 0 — Handbook selection"), ln=True)
    pdf.set_font("Helvetica", size=11)
    pdf.multi_cell(0, 6, pdf_safe(f"Course section: {course_title}"))
    pdf.multi_cell(0, 6, pdf_safe(f"Module: {plan.get('moduleName')} ({plan.get('moduleCode')})"))
    pdf.ln(2)
    pdf.set_font("Helvetica", style="B", size=13)
    pdf.cell(0, 8, pdf_safe("Step 0.5 — Lecture plan"), ln=True)
    pdf.set_font("Helvetica", size=11)
    pdf.multi_cell(0, 6, pdf_safe(f"Source: {plan.get('source')} | Lectures: {len(plan.get('lectures', []))}"))
    pdf.ln(1)
    for lecture in plan.get("lectures", []):
        line = f"{lecture.get('number')}. {lecture.get('title')} ({lecture.get('durationMinutes', 90)} min)"
        pdf.multi_cell(0, 6, pdf_safe(line))
        summary = lecture.get("summary") or ""
        if summary:
            pdf.multi_cell(0, 5, pdf_safe(f"   {summary}"))
    path.parent.mkdir(parents=True, exist_ok=True)
    pdf.output(str(path))


def persist_lecture_plan(plan: dict, course_title: str) -> dict[str, Path]:
    slug = module_slug(plan.get("moduleCode", "module"))
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    paths = {
        "json": OUTPUT_DIR / f"{slug}-lecture-plan.json",
        "markdown": OUTPUT_DIR / f"{slug}-lecture-plan.md",
        "report_pdf": OUTPUT_DIR / f"{slug}-generation-report.pdf",
        "full_run_json": OUTPUT_DIR / f"{slug}-full-run.json",
        "full_run_markdown": OUTPUT_DIR / f"{slug}-full-run.md",
        "lectures_dir": OUTPUT_DIR / slug / "lectures",
    }
    save_lecture_plan_json(plan, paths["json"])
    save_lecture_plan_markdown(plan, paths["markdown"])
    return paths


# --- Run Step 0.5 (executes when you run this cell) ---
if not (10 <= TARGET_LECTURE_COUNT <= 12):
    raise ValueError("TARGET_LECTURE_COUNT must be between 10 and 12")

LECTURE_PLAN = build_lecture_plan(
    selected_module=selected_module,
    course_context=COURSE_CONTEXT,
    handbook_section_text=handbook_section_text,
    target_count=TARGET_LECTURE_COUNT,
)
LECTURE_PLAN_PATHS = persist_lecture_plan(LECTURE_PLAN, COURSE_TITLE)

show_title("Lecture plan", level=3)
show_fields(
    {
        "source": LECTURE_PLAN.get("source"),
        "lecture_count": len(LECTURE_PLAN.get("lectures", [])),
        "json": str(LECTURE_PLAN_PATHS["json"]),
        "markdown": str(LECTURE_PLAN_PATHS["markdown"]),
        "report_pdf": str(LECTURE_PLAN_PATHS["report_pdf"]) + " (after Step 4)",
    }
)
show_table(
    [
        [lec.get("number"), lec.get("title"), lec.get("durationMinutes", 90)]
        for lec in LECTURE_PLAN.get("lectures", [])
    ],
    headers=["#", "Lecture title", "Minutes"],
)
print(f"Saved lecture plan ({LECTURE_PLAN['source']}) to {OUTPUT_DIR}")

## Steps 1–3 — Generators (outline, slides, Bloom)

Prompt templates and `generate_*` functions used by **Step 4** for each lecture in `LECTURE_PLAN`.
The old single-lecture demo runs below are disabled — use Step 4 instead.


In [4]:
OUTLINE_SYSTEM_PROMPT = """You are an expert computer science professor and curriculum designer.
Generate a structured university lecture outline in JSON format.
Focus on clear learning progression, practical examples, and CS pedagogy.
Design the outline for a slide deck of approximately {target_slides} slides (one main idea per slide).
Output ONLY valid JSON matching the requested schema."""

OUTLINE_USER_PROMPT = """Create a lecture outline with the following parameters:
Title: {title}
Description: {description}
Target slide count: {target_slides}

Return JSON with this structure:
{{
  "title": string,
  "summary": string,
  "learningObjectives": string[],
  "targetSlideCount": {target_slides},
  "estimatedStudyMinutes": number,
  "sections": [
    {{
      "title": string,
      "learningGoals": string[],
      "subsections": [
        {{
          "title": string,
          "keyPoints": string[],
          "examples": string[],
          "suggestedDiagram": string
        }}
      ],
      "exercises": string[]
    }}
  ],
  "suggestedResources": string[],
  "assessmentIdeas": string[]
}}"""


def generate_outline(title: str, description: str = "", target_slides: int = 10) -> dict:
    print(f"=== Step 1: generating outline for \'{title}\' ===")

    def _call():
        system = OUTLINE_SYSTEM_PROMPT.format(target_slides=target_slides)
        user = OUTLINE_USER_PROMPT.format(
            title=title, description=description or "N/A", target_slides=target_slides
        )
        return call_llm(system, user, temperature=0.5)

    return parse_with_retry(_call, label="outline")


In [5]:
# Single-lecture demo disabled.
# Run Step 4 to iterate outline -> slides -> Bloom for every lecture in LECTURE_PLAN.
print("Step 1 demo skipped. Use Step 4.")


## Step 2 — Slide deck generation

Takes the outline from Step 1 and turns it into a slide deck.

In [6]:
SLIDES_SYSTEM_PROMPT = """You are an expert presentation designer for university CS lectures.
Convert lecture outlines into engaging slide decks with speaker notes.
Use concise bullet points (max 5 per slide), include code snippets when relevant.
Output ONLY valid JSON."""

SLIDES_USER_PROMPT = """Convert this lecture outline into a slide deck for "{lecture_title}".
Target {target_slides} slides (\u00b12 if needed for pacing).

Each slide must map to a lecture section. Follow cognitive flow: conceptual -> visual -> code -> recap.
Avoid text-heavy slides.

Outline:
{outline_json}

Return JSON:
{{
  "slides": [
    {{
      "id": string,
      "title": string,
      "layout": "title" | "content" | "split" | "code" | "image",
      "blocks": [
        {{
          "id": string,
          "type": "heading" | "text" | "bullets" | "code" | "image" | "diagram" | "quote",
          "content": string | {{"items": string[], "language": string, "code": string, "prompt": string}}
        }}
      ],
      "speakerNotes": string,
      "metadata": {{"transition": string, "animation": string, "tags": string[]}}
    }}
  ]
}}

Guidelines:
- Start with a title slide
- One main idea per content slide
- Add diagram blocks with descriptive prompts where visuals help
- Include speaker notes for teaching tips"""


def generate_slides(outline: dict, lecture_title: str) -> dict:
    target_slides = outline.get("targetSlideCount", 10)
    print(f"=== Step 2: generating ~{target_slides} slides for \'{lecture_title}\' ===")

    def _call():
        user = SLIDES_USER_PROMPT.format(
            lecture_title=lecture_title,
            target_slides=target_slides,
            outline_json=json.dumps(outline, indent=2),
        )
        return call_llm(SLIDES_SYSTEM_PROMPT, user, temperature=0.6, max_tokens=8192)

    return parse_with_retry(_call, label="slides")


In [7]:
# Single-lecture demo disabled. See Step 4.
print("Step 2 demo skipped. Use Step 4.")


## Step 3 — Bloom's-taxonomy evaluation

Evaluates the generated deck against the outline's learning objectives, Bloom-level only (no clarity/design scoring).

In [8]:
BLOOM_LEVELS_DOC = """Revised Bloom levels (low -> high):
- remember: recall facts, terms, definitions
- understand: explain, summarize, classify, compare
- apply: use concepts in examples, implement patterns, solve standard problems
- analyze: break down designs, compare trade-offs, diagnose structure
- evaluate: critique choices, justify decisions, assess quality
- create: design solutions, propose architectures, synthesize new structures"""

EVALUATION_SYSTEM_PROMPT = """You are an expert instructional designer evaluating CS lecture slides using Revised Bloom's taxonomy only.
Analyze cognitive level alignment between learning objectives and slide content.
Be constructive and specific. Output ONLY valid JSON."""

EVALUATION_USER_PROMPT = """Evaluate this slide deck for "{lecture_title}" using Revised Bloom's taxonomy only.

Learning objectives:
{objectives_bulleted}

Slides:
{slides_json}

{bloom_levels_doc}

Return JSON:
{{
  "alignmentScore": number (0-100),
  "dominantLevel": "remember" | "understand" | "apply" | "analyze" | "evaluate" | "create",
  "distribution": {{
    "remember": number, "understand": number, "apply": number,
    "analyze": number, "evaluate": number, "create": number
  }},
  "objectiveAlignments": [
    {{
      "objective": string,
      "expectedLevel": "remember" | "understand" | "apply" | "analyze" | "evaluate" | "create",
      "observedLevel": "remember" | "understand" | "apply" | "analyze" | "evaluate" | "create",
      "status": "aligned" | "partially_aligned" | "misaligned",
      "rationale": string
    }}
  ],
  "slideTags": [{{"slideTitle": string, "primaryLevel": string, "evidence": string}}],
  "gaps": string[],
  "suggestions": string[]
}}

Rules:
- distribution values are percentages of deck cognitive demand and should sum to 100
- For each learning objective, infer expectedLevel from its wording, then compare to what slides actually teach
- status = aligned when observedLevel matches expectedLevel; partially_aligned when one level off; misaligned when two or more levels off
- For MSc-level CS lectures, flag gaps when distribution is >60% remember+understand without apply/analyze content
- Do not evaluate clarity, readability, or visual design \u2014 Bloom taxonomy only"""


def evaluate_deck(lecture_title: str, slides: list, learning_objectives: list | None = None) -> dict:
    print(f"=== Step 3: evaluating {len(slides)} slides against Bloom\'s taxonomy ===")

    def _call():
        objectives_bulleted = "\n".join(f"- {o}" for o in (learning_objectives or [])) or "N/A"
        user = EVALUATION_USER_PROMPT.format(
            lecture_title=lecture_title,
            objectives_bulleted=objectives_bulleted,
            slides_json=json.dumps(slides, indent=2),
            bloom_levels_doc=BLOOM_LEVELS_DOC,
        )
        return call_llm(EVALUATION_SYSTEM_PROMPT, user, temperature=0.3)

    return parse_with_retry(_call, label="evaluation")


In [9]:
# Single-lecture demo disabled. See Step 4.
print("Step 3 demo skipped. Use Step 4.")


## Step 4 — Run pipeline for all lectures

Iterates over `LECTURE_PLAN["lectures"]` and for **each lecture** runs:

1. outline generation
2. slide deck generation
3. Bloom evaluation

Artifacts:
- per-lecture JSON under `output/course-generation/{module}/lectures/`
- combined `{module}-full-run.json` and `{module}-full-run.md`
- updated `{module}-generation-report.pdf` with lecture plan + all Bloom scores

Set `MAX_LECTURES_TO_PROCESS = 1` for a quick smoke test.


In [ ]:
# --- Step 4 settings ---
SLIDES_PER_LECTURE = 18  # target slides per lecture deck
MAX_LECTURES_TO_PROCESS = 1  # e.g. 1 for quick test; None = all lectures


def build_lecture_description(lecture: dict, module: dict, course_context: str) -> str:
    topics = lecture.get("topics") or []
    lines = [
        f"Module: {module['name']} ({module['code']})",
        f"Lecture number: {lecture.get('number')}",
        f"Lecture title: {lecture.get('title')}",
        f"Duration: {lecture.get('durationMinutes', 90)} minutes",
    ]
    if lecture.get("summary"):
        lines.append(f"Lecture summary: {lecture['summary']}")
    if topics:
        lines.append("Lecture topics: " + "; ".join(topics))
    lines.extend(["", "Handbook / module context:", course_context[:8000]])
    return "\n".join(lines)


def lecture_artifact_paths(base_dir: Path, lecture_number: int) -> dict[str, Path]:
    prefix = base_dir / f"lecture-{lecture_number:02d}"
    return {
        "outline": Path(f"{prefix}-outline.json"),
        "deck": Path(f"{prefix}-deck.json"),
        "bloom": Path(f"{prefix}-bloom.json"),
    }


def save_full_run_markdown(run_bundle: dict, path: Path) -> None:
    plan = run_bundle["lecturePlan"]
    lines = [
        f"# Full run — {plan.get('moduleName')}",
        "",
        f"- **Module code:** {plan.get('moduleCode')}",
        f"- **Lectures processed:** {len(run_bundle['lectures'])}",
        f"- **Generated at:** {run_bundle['generatedAt']}",
        "",
        "## Bloom summary",
        "",
        "| # | Lecture | Alignment | Dominant | Slides |",
        "|---|---------|-----------|----------|--------|",
    ]
    for item in run_bundle["lectures"]:
        ev = item.get("evaluation") or {}
        lines.append(
            f"| {item['number']} | {item['title']} | {ev.get('alignmentScore', '—')} | {ev.get('dominantLevel', '—')} | {item.get('slideCount', '—')} |"
        )
    lines.append("")
    for item in run_bundle["lectures"]:
        lines.append(f"## Lecture {item['number']}. {item['title']}")
        outline = item.get("outline") or {}
        ev = item.get("evaluation") or {}
        lines.append(f"- **Outline summary:** {outline.get('summary', '—')}")
        lines.append(f"- **Bloom alignment:** {ev.get('alignmentScore', '—')}")
        gaps = ev.get("gaps") or []
        if gaps:
            lines.append("- **Gaps:**")
            lines.extend(f"  - {g}" for g in gaps[:5])
        lines.append("")
    path.write_text("\n".join(lines), encoding="utf-8")


def save_full_generation_report_pdf(run_bundle: dict, course_title: str, path: Path) -> None:
    FPDF = _ensure_fpdf()
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)

    def heading(text: str, size: int = 13):
        pdf.ln(2)
        pdf.set_font("Helvetica", style="B", size=size)
        pdf.multi_cell(0, 8, pdf_safe(text), new_x="LMARGIN", new_y="NEXT")
        pdf.set_font("Helvetica", size=11)

    def body(text: str):
        pdf.multi_cell(0, 6, pdf_safe(text), new_x="LMARGIN", new_y="NEXT")

    plan = run_bundle["lecturePlan"]
    pdf.add_page()
    heading("Course Generation Algorithm — Full Report", 16)
    body(f"Generated: {run_bundle['generatedAt']}")
    body(f"Course section: {course_title}")
    body(f"Module: {plan.get('moduleName')} ({plan.get('moduleCode')})")

    heading("Step 0.5 — Lecture plan")
    body(f"Source: {plan.get('source')} | Lectures: {len(plan.get('lectures', []))}")
    for lec in plan.get("lectures", []):
        body(f"{lec.get('number')}. {lec.get('title')} ({lec.get('durationMinutes', 90)} min)")

    for item in run_bundle["lectures"]:
        pdf.add_page()
        heading(f"Lecture {item['number']}: {item['title']}")
        outline = item.get("outline") or {}
        ev = item.get("evaluation") or {}
        body(f"Slides: {item.get('slideCount', 0)}")
        body(f"Outline summary: {outline.get('summary', '—')}")
        objectives = outline.get("learningObjectives") or []
        if objectives:
            body("Learning objectives:")
            for obj in objectives[:6]:
                body(f"  - {obj}")
        body(f"Bloom alignment: {ev.get('alignmentScore', '—')} | Dominant: {ev.get('dominantLevel', '—')}")
        gaps = ev.get("gaps") or []
        if gaps:
            body("Gaps:")
            for gap in gaps[:5]:
                body(f"  - {gap}")
        suggestions = ev.get("suggestions") or []
        if suggestions:
            body("Suggestions:")
            for sug in suggestions[:5]:
                body(f"  - {sug}")

    path.parent.mkdir(parents=True, exist_ok=True)
    pdf.output(str(path))


def run_pipeline_for_all_lectures(
    lecture_plan: dict,
    module: dict,
    course_context: str,
    course_title: str,
    paths: dict[str, Path],
    *,
    slides_per_lecture: int = 18,
    max_lectures: int | None = None,
) -> dict:
    lectures = lecture_plan.get("lectures", [])
    if max_lectures is not None:
        lectures = lectures[:max_lectures]

    lectures_dir = paths["lectures_dir"]
    lectures_dir.mkdir(parents=True, exist_ok=True)

    run_items: list[dict] = []
    print(f"=== Step 4: processing {len(lectures)} lecture(s) ===")

    for lecture in lectures:
        number = int(lecture.get("number") or len(run_items) + 1)
        title = lecture.get("title") or f"Lecture {number}"
        full_title = f"Lecture {number}. {title}"
        print(f"\n--- [{number}/{len(lectures)}] {full_title} ---")

        description = build_lecture_description(lecture, module, course_context)
        outline = generate_outline(title=full_title, description=description, target_slides=slides_per_lecture)
        deck = generate_slides(outline, lecture_title=outline.get("title", full_title))
        slides = deck.get("slides", [])
        evaluation = evaluate_deck(
            lecture_title=outline.get("title", full_title),
            slides=slides,
            learning_objectives=outline.get("learningObjectives"),
        )

        artifact_paths = lecture_artifact_paths(lectures_dir, number)
        artifact_paths["outline"].write_text(json.dumps(outline, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        artifact_paths["deck"].write_text(json.dumps(deck, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        artifact_paths["bloom"].write_text(json.dumps(evaluation, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

        run_items.append(
            {
                "number": number,
                "title": title,
                "outline": outline,
                "deck": deck,
                "evaluation": evaluation,
                "slideCount": len(slides),
                "artifacts": {k: str(v) for k, v in artifact_paths.items()},
            }
        )

        show_title(f"Lecture {number} — Bloom {evaluation.get('alignmentScore')}/100", level=4)

    run_bundle = {
        "generatedAt": datetime.now(timezone.utc).isoformat(),
        "courseTitle": course_title,
        "lecturePlan": lecture_plan,
        "lectures": run_items,
    }

    paths["full_run_json"].write_text(json.dumps(run_bundle, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    save_full_run_markdown(run_bundle, paths["full_run_markdown"])
    save_full_generation_report_pdf(run_bundle, course_title, paths["report_pdf"])

    return run_bundle


FULL_RUN = run_pipeline_for_all_lectures(
    lecture_plan=LECTURE_PLAN,
    module=selected_module,
    course_context=COURSE_CONTEXT,
    course_title=COURSE_TITLE,
    paths=LECTURE_PLAN_PATHS,
    slides_per_lecture=SLIDES_PER_LECTURE,
    max_lectures=MAX_LECTURES_TO_PROCESS,
)

show_title("Full run complete", level=3)
show_fields(
    {
        "lectures_processed": len(FULL_RUN["lectures"]),
        "full_run_json": str(LECTURE_PLAN_PATHS["full_run_json"]),
        "full_run_markdown": str(LECTURE_PLAN_PATHS["full_run_markdown"]),
        "report_pdf": str(LECTURE_PLAN_PATHS["report_pdf"]),
        "lectures_dir": str(LECTURE_PLAN_PATHS["lectures_dir"]),
    }
)
show_table(
    [
        [
            item["number"],
            item["title"],
            item["evaluation"].get("alignmentScore"),
            item["evaluation"].get("dominantLevel"),
            item["slideCount"],
        ]
        for item in FULL_RUN["lectures"]
    ],
    headers=["#", "Lecture", "Bloom", "Dominant", "Slides"],
)
print("Saved full run artifacts.")
